In [ ]:
"""
Test exhaustivo de validación Pipeline RITMO (Pasos 1-3) - WEATHER DATASET.

Valida implementación completa de:
- Paso 1: Normalización RevIN (reversible)
- Paso 2: Entrenamiento HMM (Baum-Welch con emisiones gaussianas)
- Paso 3: Tokenización Viterbi (estados óptimos)

Dataset: Weather (temperatura wet-bulb)
- Dominio: Datos climáticos (diferente a temperatura transformador ETT)
- Regímenes: Estaciones climáticas (verano/otoño/invierno/primavera)
- Características: Cambios de régimen estacionales muy claros
- Tamaño: Mayor que ETT datasets

Configuraciones de prueba:
- K ∈ {3, 4, 5, 6, 7, 8, 9} estados HMM (loop continuo)
- seeds ∈ {42, 123, 456} para robustez
- data_configs: full, half (timesteps variables según dataset)
- max_iter: 300 iteraciones (convergencia 100%)
Total: 7 × 3 × 2 = 42 configuraciones

Métricas reportadas:
- RevIN: MSE reconstrucción
- HMM: Log-likelihood, AIC, BIC, convergencia
- Viterbi: Ratio compresión, entropía, segmentos
- Validaciones: Matriz estocástica, parámetros razonables, estados no degenerados

Salida: Tabla comparativa con recomendación automática de K óptimo según AIC/BIC.
"""

import numpy as np
from data_provider.data_loader import Dataset_Custom
from utils.revin import RevINNormalizer
from hmm import baum_welch, viterbi_decode


class Args:
    """Mock args requerido por Dataset_Custom."""
    augmentation_ratio = 0


def load_weather(root_path='dataset/weather/',
                 data_path='weather.csv',
                 target='OT',
                 data_size=None):
    """
    Carga Weather univariado sin normalización.

    Args:
        root_path: Directorio datasets
        data_path: Archivo CSV (weather.csv)
        target: Columna objetivo (OT: última columna, temperatura wet-bulb)
        data_size: Limitar train a N timesteps (None: usar completo)

    Returns:
        dict: {'train': array[T], 'val': array[V], 'test': array[Te]}
    """
    dataset_train = Dataset_Custom(
        args=Args(),
        root_path=root_path,
        flag='train',
        size=None,
        features='S',
        data_path=data_path,
        target=target,
        scale=False,
        timeenc=0,
        freq='h'
    )

    dataset_val = Dataset_Custom(
        args=Args(),
        root_path=root_path,
        flag='val',
        size=None,
        features='S',
        data_path=data_path,
        target=target,
        scale=False,
        timeenc=0,
        freq='h'
    )

    dataset_test = Dataset_Custom(
        args=Args(),
        root_path=root_path,
        flag='test',
        size=None,
        features='S',
        data_path=data_path,
        target=target,
        scale=False,
        timeenc=0,
        freq='h'
    )

    train = dataset_train.data_x.flatten()
    val = dataset_val.data_x.flatten()
    test = dataset_test.data_x.flatten()

    if data_size is not None:
        train = train[:data_size]

    return {'train': train, 'val': val, 'test': test}


def validate_revin(data_original, data_normalized, normalizer, split='train', threshold=1e-6):
    """
    Valida reversibilidad normalización RevIN.

    Comprueba que denorm(norm(X)) ≈ X con MSE < threshold.

    Args:
        data_original: Datos raw
        data_normalized: Datos después de RevIN
        normalizer: Instancia RevINNormalizer fitted
        split: Split a validar ('train', 'val', 'test')
        threshold: MSE máximo aceptable

    Returns:
        (success: bool, mse: float)
    """
    success, mse = normalizer.validate_reconstruction(
        original=data_original,
        normalized=data_normalized,
        split=split,
        threshold=threshold
    )
    return success, mse


def validate_hmm_params(params, K):
    """
    Valida validez matemática parámetros HMM λ = (A, π, μ, σ).

    Checks de validez:
    - Propiedades estocásticas: filas A suman 1, π suma 1
    - Propiedades físicas: σ > 0, μ finitos
    - Propiedades dimensionales: shapes correctos
    - Propiedades de calidad: estados diferenciados, sin degeneración

    Args:
        params: dict con A[K,K], pi[K], mu[K], sigma[K]
        K: Número de estados

    Returns:
        dict: {check_name: bool} para cada validación
    """
    checks = {}

    row_sums = np.sum(params['A'], axis=1)
    checks['matrix_stochastic'] = np.allclose(row_sums, 1.0, atol=1e-4)
    checks['pi_stochastic'] = np.isclose(np.sum(params['pi']), 1.0, atol=1e-4)
    checks['sigma_positive'] = np.all(params['sigma'] > 0)
    checks['mu_finite'] = np.all(np.isfinite(params['mu']))
    checks['dims_correct'] = (
        params['A'].shape == (K, K) and
        params['pi'].shape == (K,) and
        params['mu'].shape == (K,) and
        params['sigma'].shape == (K,)
    )
    checks['no_degenerate_states'] = True
    checks['no_extreme_persistence'] = np.all(np.diag(params['A']) < 0.99)
    checks['sigma_reasonable'] = np.all((params['sigma'] > 0.05) & (params['sigma'] < 10.0))
    checks['mu_spread'] = (params['mu'].max() - params['mu'].min()) > 0.5

    return checks


def validate_viterbi(states_pred, K, data_norm):
    """
    Valida output de Viterbi Q* = argmax P(Q|O,λ).

    Checks:
    - Dimensiones: longitud match con observaciones
    - Rango: estados ∈ [0, K-1]
    - Finitud: sin NaN/Inf
    - Segmentación: al menos 1 cambio de estado

    Args:
        states_pred: array[T] con estados predichos
        K: Número de estados HMM
        data_norm: Datos normalizados (para validar longitud)

    Returns:
        dict: {check_name: bool}
    """
    checks = {}

    checks['length_match'] = len(states_pred) == len(data_norm)
    checks['states_in_range'] = np.all((states_pred >= 0) & (states_pred < K))
    checks['states_finite'] = np.all(np.isfinite(states_pred))

    n_segments = np.sum(np.diff(states_pred) != 0) + 1
    checks['has_segments'] = n_segments >= 1

    return checks


def run_single_config(K, seed, data_config, verbose=False):
    """
    Ejecuta pipeline completo para una configuración (K, seed, data_size).

    Pipeline:
    1. Cargar Weather
    2. Normalizar con RevIN
    3. Entrenar HMM (Baum-Welch)
    4. Tokenizar (Viterbi)
    5. Calcular métricas (AIC, BIC, compresión, entropía)
    6. Validar todos los pasos

    Args:
        K: Número de estados HMM
        seed: Random seed para reproducibilidad
        data_config: dict con 'name' y 'size' (timesteps)
        verbose: Si True, imprime progreso

    Returns:
        dict con todas las métricas y validaciones
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"CONFIG: K={K}, seed={seed}, data={data_config['name']}")
        print(f"{'='*60}")

    data = load_weather(data_size=data_config['size'])
    train_size = len(data['train'])

    if verbose:
        print(f"Train size: {train_size} timesteps")

    normalizer = RevINNormalizer(num_features=1, eps=1e-5, affine=False)
    data_norm = normalizer.fit_transform(
        train_data=data['train'],
        val_data=data['val'],
        test_data=data['test']
    )

    revin_success, revin_mse = validate_revin(
        data['train'],
        data_norm['train'],
        normalizer,
        split='train'
    )

    if verbose:
        status = "✓" if revin_success else "✗"
        print(f"{status} RevIN MSE: {revin_mse:.2e}")

    params = baum_welch(
        data_norm['train'],
        K=K,
        max_iter=300,
        epsilon=1e-4,
        random_state=seed,
        verbose=False
    )

    hmm_checks = validate_hmm_params(params, K)

    if verbose:
        status = "✓" if params['converged'] else "⚠"
        print(f"{status} HMM convergió: {params['converged']} ({params['n_iter']} iter)")
        for check_name, check_value in hmm_checks.items():
            status = "✓" if check_value else "✗"
            print(f"{status} {check_name}: {check_value}")

    states_pred, log_likelihood = viterbi_decode(
        data_norm['train'],
        params['A'],
        params['pi'],
        params['mu'],
        params['sigma']
    )

    viterbi_checks = validate_viterbi(states_pred, K, data_norm['train'])

    unique_states = len(np.unique(states_pred))
    n_segments = np.sum(np.diff(states_pred) != 0) + 1
    n_tokens_llm = len(states_pred)
    compression_ratio = len(states_pred) / n_segments
    avg_segment_duration = len(states_pred) / n_segments

    viterbi_checks['no_degenerate_states'] = unique_states >= K * 0.7
    hmm_checks['no_degenerate_states'] = unique_states >= K * 0.7

    num_params = K**2 + 2*K
    AIC = -2 * log_likelihood + 2 * num_params
    BIC = -2 * log_likelihood + np.log(len(states_pred)) * num_params

    state_counts = np.bincount(states_pred, minlength=K)
    state_distribution = state_counts / len(states_pred)
    state_entropy = -np.sum(state_distribution * np.log(state_distribution + 1e-10))

    if verbose:
        status = "✓" if not np.isnan(log_likelihood) else "✗"
        print(f"{status} Log-likelihood: {log_likelihood:.2f}")
        print(f"  AIC: {AIC:.2f}, BIC: {BIC:.2f}")
        print(f"  Estados activos: {unique_states}/{K}")
        print(f"  Entropía estados: {state_entropy:.3f}")
        print(f"  Segmentos: {n_segments}")
        print(f"  Compresión: {compression_ratio:.1f}x")
        print(f"  Persistencia: {avg_segment_duration:.1f} timesteps")

    result = {
        'K': K,
        'seed': seed,
        'data_config': data_config['name'],
        'train_size': train_size,
        'revin_success': revin_success,
        'revin_mse': revin_mse,
        'hmm_converged': params['converged'],
        'hmm_n_iter': params['n_iter'],
        'hmm_checks': hmm_checks,
        'log_likelihood': log_likelihood,
        'AIC': AIC,
        'BIC': BIC,
        'unique_states': unique_states,
        'n_tokens_llm': n_tokens_llm,
        'n_segments': n_segments,
        'compression_ratio': compression_ratio,
        'avg_segment_duration': avg_segment_duration,
        'state_entropy': state_entropy,
        'viterbi_checks': viterbi_checks,
        'data_raw': data['train'],
        'data_norm': data_norm['train'],
        'states': states_pred,
        'params': params
    }

    return result


def print_summary(results):
    """
    Imprime reporte consolidado de todas las configuraciones.

    Secciones del reporte:
    1. Validación RevIN (paso 1)
    2. Validación HMM (paso 2)
    3. Validación Viterbi (paso 3)
    4. Tabla comparativa por K
    5. Recomendación automática K óptimo (AIC/BIC)
    6. Verificación final

    Args:
        results: list de dicts, uno por configuración ejecutada
    """
    print("\n" + "="*80)
    print("REPORTE EXHAUSTIVO - VALIDACIÓN PASOS 1-3")
    print("="*80)

    total_configs = len(results)
    revin_passed = sum(1 for r in results if r['revin_success'])
    hmm_converged = sum(1 for r in results if r['hmm_converged'])

    hmm_checks_all = [r['hmm_checks'] for r in results]
    check_names = list(hmm_checks_all[0].keys())

    print(f"\nTotal configuraciones: {total_configs}")
    print(f"\n{'='*80}")
    print("PASO 1: REVIN NORMALIZACIÓN")
    print(f"{'='*80}")
    print(f"✓ Pasadas: {revin_passed}/{total_configs} ({100*revin_passed/total_configs:.1f}%)")

    mse_values = [r['revin_mse'] for r in results]
    print(f"  MSE min: {min(mse_values):.2e}")
    print(f"  MSE max: {max(mse_values):.2e}")
    print(f"  MSE mean: {np.mean(mse_values):.2e}")

    print(f"\n{'='*80}")
    print("PASO 2: HMM ENTRENAMIENTO")
    print(f"{'='*80}")
    print(f"✓ Convergió: {hmm_converged}/{total_configs} ({100*hmm_converged/total_configs:.1f}%)")

    for check_name in check_names:
        check_passed = sum(1 for r in results if r['hmm_checks'][check_name])
        status = "✓" if check_passed == total_configs else "✗"
        print(f"{status} {check_name}: {check_passed}/{total_configs}")

    print(f"\n{'='*80}")
    print("PASO 3: VITERBI TOKENIZACIÓN")
    print(f"{'='*80}")

    viterbi_checks_all = [r['viterbi_checks'] for r in results]
    viterbi_check_names = list(viterbi_checks_all[0].keys())

    for check_name in viterbi_check_names:
        check_passed = sum(1 for r in results if r['viterbi_checks'][check_name])
        status = "✓" if check_passed == total_configs else "✗"
        print(f"{status} {check_name}: {check_passed}/{total_configs}")

    print(f"\n{'='*90}")
    print("TABLA COMPARATIVA - ANÁLISIS POR NÚMERO DE ESTADOS (K)")
    print(f"{'='*90}")

    print(f"\n{' ':<6} {' ':<10} {' ':<8} {' ':<12} {' ':<12} {' ':<10} {' ':<10}")
    print(f"{'K':<6} {'LL (mean)':<10} {'Conv':<8} {'AIC (mean)':<12} {'BIC (mean)':<12} {'Compresión':<10} {'Estados'}")
    print("-" * 90)

    K_values = sorted(set(r['K'] for r in results))
    for K in K_values:
        K_results = [r for r in results if r['K'] == K]
        ll_values = [r['log_likelihood'] for r in K_results if not np.isnan(r['log_likelihood'])]
        aic_values = [r['AIC'] for r in K_results if not np.isnan(r['AIC'])]
        bic_values = [r['BIC'] for r in K_results if not np.isnan(r['BIC'])]
        compression_values = [r['compression_ratio'] for r in K_results]
        states_values = [r['unique_states'] for r in K_results]
        converged = sum(1 for r in K_results if r['hmm_converged'])
        total = len(K_results)

        print(f"{K:<6} {np.mean(ll_values):<10.1f} {converged}/{total:<5} "
              f"{np.mean(aic_values):<12.1f} {np.mean(bic_values):<12.1f} "
              f"{np.mean(compression_values):<10.1f}x {np.mean(states_values):.1f}/{K}")

    print(f"\n{'='*90}")
    print("RECOMENDACIÓN AUTOMÁTICA (criterios AIC/BIC)")
    print(f"{'='*90}")

    best_bic_result = min(results, key=lambda r: r['BIC'] if not np.isnan(r['BIC']) else float('inf'))
    best_aic_result = min(results, key=lambda r: r['AIC'] if not np.isnan(r['AIC']) else float('inf'))

    print(f"\n✓ Mejor K según BIC: K={best_bic_result['K']}")
    print(f"  BIC: {best_bic_result['BIC']:.2f}")
    print(f"  Log-likelihood: {best_bic_result['log_likelihood']:.2f}")
    print(f"  Compresión: {best_bic_result['compression_ratio']:.1f}x")
    print(f"  Estados activos: {best_bic_result['unique_states']}/{best_bic_result['K']}")

    print(f"\n✓ Mejor K según AIC: K={best_aic_result['K']}")
    print(f"  AIC: {best_aic_result['AIC']:.2f}")
    print(f"  Log-likelihood: {best_aic_result['log_likelihood']:.2f}")
    print(f"  Compresión: {best_aic_result['compression_ratio']:.1f}x")
    print(f"  Estados activos: {best_aic_result['unique_states']}/{best_aic_result['K']}")

    print(f"\n{'='*80}")
    print("VERIFICACIÓN FINAL")
    print(f"{'='*80}")

    all_passed = (
        revin_passed == total_configs and
        all(all(r['hmm_checks'].values()) for r in results) and
        all(all(r['viterbi_checks'].values()) for r in results)
    )

    if all_passed:
        print("✓ TODOS LOS TESTS PASADOS - PASOS 1-3 VALIDADOS AL 100%")
    else:
        print("✗ ALGUNOS TESTS FALLARON - REVISAR CONFIGURACIONES")

    print("="*80 + "\n")


if __name__ == "__main__":
    print("="*80)
    print("TEST EXHAUSTIVO HMM - WEATHER + RevIN")
    print("="*80)
    print("\nValidando Pipeline RITMO (Pasos 1-3):")
    print("  1. Normalización RevIN")
    print("  2. Entrenamiento HMM (Baum-Welch)")
    print("  3. Tokenización (Viterbi)")
    print("\nDataset: Weather (temperatura wet-bulb)")
    print("  - Regímenes estacionales claros")
    print("  - Dominio climático diferente a ETT")
    print("  - Test de cambios de régimen estacionales")

    K_values = list(range(3, 10))
    seeds = [42, 123, 456]
    data_configs = [
        {'name': 'full', 'size': None},
        {'name': 'half', 'size': 18000}
    ]

    total_configs = len(K_values) * len(seeds) * len(data_configs)
    print(f"\nTotal configuraciones: {total_configs}")
    print(f"  K values: {K_values}")
    print(f"  Seeds: {seeds}")
    print(f"  Data configs: {[d['name'] for d in data_configs]}")
    print(f"  Max iterations: 300 (convergencia 100%)")

    results = []
    config_idx = 0

    for K in K_values:
        for seed in seeds:
            for data_config in data_configs:
                config_idx += 1
                print(f"\n[{config_idx}/{total_configs}] ", end='')

                result = run_single_config(K, seed, data_config, verbose=True)
                results.append(result)

    print_summary(results)

    print("\n" + "="*80)
    print("✓ TEST EXHAUSTIVO COMPLETADO")
    print("="*80)
    print("\nPróximos pasos:")
    print("  - Si todos los tests pasaron → Avanzar a Fase 4 (Embeddings)")
    print("  - Si algunos tests fallaron → Investigar configuraciones problemáticas")
    print("="*80 + "\n")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

In [ ]:
"""
DASHBOARD COMPLETO - Validación Pipeline RITMO (Fases 1-3)
Dataset: WEATHER (temperatura wet-bulb)
Visualización con TODOS los timesteps + leyenda de estados
"""

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.patches import Rectangle, Patch
import numpy as np
import os

sns.set_style("whitegrid")
plt.rcParams['font.size'] = 9
plt.rcParams['figure.dpi'] = 100

os.makedirs('./pic', exist_ok=True)

print("\n" + "="*80)
print("GENERANDO DASHBOARD COMPLETO - WEATHER - TODOS LOS TIMESTEPS")
print("="*80)

def get_segments(states):
    segments = []
    current_state = states[0]
    start_idx = 0
    
    for i in range(1, len(states)):
        if states[i] != current_state:
            segments.append((start_idx, i, current_state))
            current_state = states[i]
            start_idx = i
    
    segments.append((start_idx, len(states), current_state))
    return segments


def create_state_legend(ax, K, mu, sigma, A, cmap):
    ax.axis('off')
    x_positions = np.linspace(0.02, 0.98, K)
    
    for k in range(K):
        color = cmap(k)
        x = x_positions[k]
        text = f"S{k}: μ={mu[k]:.2f}, σ={sigma[k]:.2f}, P(stay)={A[k,k]*100:.0f}%"
        ax.text(x, 0.5, text, ha='center', va='center', fontsize=8,
                transform=ax.transAxes,
                bbox=dict(boxstyle='round,pad=0.3', facecolor=color, 
                         alpha=0.7, edgecolor='black', linewidth=0.5))


configs_to_plot = []
for K in range(3, 10):
    result = [r for r in results if r['K']==K and r['seed']==42 
              and r['data_config']=='full'][0]
    configs_to_plot.append(result)

data_raw_full = configs_to_plot[0]['data_raw']
data_norm_full = configs_to_plot[0]['data_norm']
T = len(data_raw_full)

print(f"Datos preparados: {len(configs_to_plot)} configuraciones (K=3-9)")
print(f"Timesteps visualizados: {T} (TODOS)")

n_K = 7
fig = plt.figure(figsize=(50, 60))

gs_main = fig.add_gridspec(16, 2, width_ratios=[3, 1], 
                           height_ratios=[1, 1] + [1, 0.15]*7,
                           hspace=0.3, wspace=0.2)

print("\nGenerando dashboard...")

ax_orig = fig.add_subplot(gs_main[0, 0])
ax_orig.plot(data_raw_full, color='#9b59b6', linewidth=0.8, alpha=0.9)
ax_orig.set_xlabel('Timestep', fontweight='bold')
ax_orig.set_ylabel('Temperatura Wet-Bulb', fontweight='bold')
ax_orig.set_title(f'Serie Original Weather (temperatura wet-bulb) - {T} timesteps completos',
                 fontweight='bold', fontsize=12, pad=10)
ax_orig.grid(alpha=0.3)

mean_raw = np.mean(data_raw_full)
std_raw = np.std(data_raw_full)
ax_orig.text(0.01, 0.98, f'μ={mean_raw:.2f}\nσ={std_raw:.2f}\nT={T}',
            transform=ax_orig.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='#ebdef0', alpha=0.8))

ax_hist_orig = fig.add_subplot(gs_main[0, 1])
ax_hist_orig.hist(data_raw_full, bins=50, density=True, alpha=0.7, 
                  color='#9b59b6', edgecolor='black', linewidth=0.3)
ax_hist_orig.set_xlabel('Valor', fontweight='bold')
ax_hist_orig.set_ylabel('Densidad', fontweight='bold')
ax_hist_orig.set_title('Distribución\nSerie Original', fontweight='bold', fontsize=10)
ax_hist_orig.grid(alpha=0.3)

ax_norm = fig.add_subplot(gs_main[1, 0])
ax_norm.plot(data_norm_full, color='#2ecc71', linewidth=0.8, alpha=0.9)
ax_norm.axhline(y=1, color='red', linestyle='--', alpha=0.3, linewidth=1)
ax_norm.axhline(y=-1, color='red', linestyle='--', alpha=0.3, linewidth=1)
ax_norm.axhline(y=0, color='black', linestyle='-', alpha=0.2, linewidth=0.5)
ax_norm.fill_between(range(T), -1, 1, alpha=0.1, color='gray')

ax_norm.set_xlabel('Timestep', fontweight='bold')
ax_norm.set_ylabel('Valor Normalizado', fontweight='bold')
ax_norm.set_title(f'Serie Normalizada RevIN - {T} timesteps',
                 fontweight='bold', fontsize=12, pad=10)
ax_norm.grid(alpha=0.3)

mean_norm = np.mean(data_norm_full)
std_norm = np.std(data_norm_full)
ax_norm.text(0.01, 0.98, f'μ={mean_norm:.4f}\nσ={std_norm:.4f}\n✓ RevIN OK',
            transform=ax_norm.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

ax_hist_norm = fig.add_subplot(gs_main[1, 1])
ax_hist_norm.hist(data_norm_full, bins=50, density=True, alpha=0.7,
                  color='#2ecc71', edgecolor='black', linewidth=0.3, label='Datos')

x_theo = np.linspace(data_norm_full.min()-0.5, data_norm_full.max()+0.5, 100)
y_theo = (1/(std_norm*np.sqrt(2*np.pi))) * np.exp(-0.5 * ((x_theo-mean_norm)/std_norm)**2)
ax_hist_norm.plot(x_theo, y_theo, 'r-', linewidth=2, label=f'N({mean_norm:.2f},{std_norm:.2f})')

ax_hist_norm.set_xlabel('Valor', fontweight='bold')
ax_hist_norm.set_ylabel('Densidad', fontweight='bold')
ax_hist_norm.set_title('Distribución\nRevIN', fontweight='bold', fontsize=10)
ax_hist_norm.legend(fontsize=7)
ax_hist_norm.grid(alpha=0.3)

for idx, result in enumerate(configs_to_plot):
    K = result['K']
    row_series = 2 + idx * 2
    row_legend = 2 + idx * 2 + 1
    
    print(f"  Generando K={K}...")
    
    data_norm = result['data_norm']
    states = result['states']
    mu = result['params']['mu']
    sigma = result['params']['sigma']
    A = result['params']['A']
    
    cmap = plt.cm.get_cmap('tab10', K)
    
    ax_series = fig.add_subplot(gs_main[row_series, 0])
    
    segments = get_segments(states)
    for start, end, k in segments:
        color = cmap(k)
        ax_series.plot(range(start, end), data_norm[start:end],
                      color=color, linewidth=0.8, alpha=0.9)
        ax_series.axvspan(start, end, alpha=0.15, color=color)
    
    title = (f"K={K} | AIC={result['AIC']:.1f} | BIC={result['BIC']:.1f} | "
             f"Compresión={result['compression_ratio']:.1f}x | "
             f"Segmentos={result['n_segments']} | Entropía={result['state_entropy']:.2f}")
    
    ax_series.set_xlabel('Timestep', fontweight='bold')
    ax_series.set_ylabel('Valor Norm.', fontweight='bold')
    ax_series.set_title(title, fontweight='bold', fontsize=10, pad=5)
    ax_series.grid(alpha=0.3)
    ax_series.set_xlim(0, T)
    
    ax_legend = fig.add_subplot(gs_main[row_legend, 0])
    create_state_legend(ax_legend, K, mu, sigma, A, cmap)
    
    ax_matrix = fig.add_subplot(gs_main[row_series:row_series+2, 1])
    
    im = ax_matrix.imshow(A, cmap='YlOrRd', vmin=0, vmax=1, aspect='equal')
    
    for i in range(K):
        for j in range(K):
            text_color = 'white' if A[i,j] > 0.5 else 'black'
            ax_matrix.text(j, i, f'{A[i,j]:.2f}', ha='center', va='center',
                          color=text_color, fontsize=7, fontweight='bold')
    
    for k in range(K):
        rect = Rectangle((k-0.5, k-0.5), 1, 1, fill=False, 
                        edgecolor='blue', linewidth=2)
        ax_matrix.add_patch(rect)
    
    ax_matrix.set_xticks(range(K))
    ax_matrix.set_yticks(range(K))
    ax_matrix.set_xticklabels([f'S{i}' for i in range(K)], fontsize=8)
    ax_matrix.set_yticklabels([f'S{i}' for i in range(K)], fontsize=8)
    ax_matrix.set_xlabel('Estado siguiente', fontweight='bold', fontsize=9)
    ax_matrix.set_ylabel('Estado actual', fontweight='bold', fontsize=9)
    ax_matrix.set_title(f'Matriz A ({K}x{K})', fontweight='bold', fontsize=10)
    
    cbar = plt.colorbar(im, ax=ax_matrix, fraction=0.046, pad=0.04)
    cbar.set_label('P(j|i)', fontsize=8)

plt.tight_layout()
output_path = './pic/dashboard_completo_validacion_ritmo_weather.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"\n✓ Dashboard guardado en: {output_path}")
print(f"  Resolución: 50x60 pulgadas @ 150 DPI")
plt.show()